# 🛰️ SatQuest — Continued SFT + Deploy to HuggingFace



## 📦 Step 1 — Install Dependencies

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-80GB
VRAM    : 85.1 GB


## 🔧 Step 2 — Configuration (edit only the token here)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  ⚙️  CONFIGURATION — only change HF_TOKEN below, everything else is set
# ═══════════════════════════════════════════════════════════════════════

# ── Model source ( account — load only, don't touch) ─────────────
SOURCE_MODEL  = ""

# ── YOUR HuggingFace account — trained model will be deployed here ────────
HF_USERNAME   = "xxxx"                           # your HF username
HF_REPO_NAME  = "satquest-qwen2vl-sft-v2"          # new repo name

# ⚠️  PASTE YOUR WRITE TOKEN HERE
# Get it from: https://huggingface.co/settings/tokens
# Click: New token → Type: Write → Copy
HF_TOKEN      = ""      # ← REPLACE THIS

REPO_PRIVATE  = False   # False = public repo

# ── What to upload to HuggingFace ────────────────────────────────────────
UPLOAD_ADAPTER = True   # LoRA adapter only (~116 MB) — fast upload
UPLOAD_MERGED  = True   # Full merged model (~4 GB)  — best for backend

# ── Paths (no need to change) ────────────────────────────────────────────
IMAGE_DIR     = "/content/HRVQA/images"            # {image_id}.png
OUTPUT_DIR    = "/content/satquest_ckpts"
ADAPTER_SAVE  = "/content/satquest_adapter"
MERGED_SAVE   = "/content/satquest_merged"
DRIVE_BACKUP  = "/content/drive/MyDrive/satquest_continued_sft_adapter"
DRIVE_MERGED  = "/content/drive/MyDrive/satquest_merged_model"

# ── Training hyperparameters ─────────────────────────────────────────────
MAX_SEQ_LEN   = 512
BATCH_SIZE    = 2
GRAD_ACCUM    = 8      # effective batch = 16
LEARNING_RATE = 1e-4
NUM_EPOCHS    = 2
MAX_STEPS     = -1     # -1 = full epochs; set 100 for a quick test run
WARMUP_RATIO  = 0.05
WEIGHT_DECAY  = 0.01
LR_SCHEDULER  = "cosine"
SAVE_STEPS    = 200
LOGGING_STEPS = 10
EVAL_STEPS    = 200
MAX_GRAD_NORM = 1.0
BF16          = True   # A100 supports bfloat16
SEED          = 42

print("✅ Configuration loaded")
print(f"   Source model   : {SOURCE_MODEL}")
print(f"   Deploy to      : xxxx/{HF_REPO_NAME}  (adapter)")
print(f"   Deploy to      : xxxxx/{HF_REPO_NAME}-merged  (full model)")
print(f"   Effective batch: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}")

✅ Configuration loaded
   Source model   : Abdullah-123/qwen2vl-2b-hrvqa-lora
   Deploy to      : Humaix/satquest-qwen2vl-sft-v2  (adapter)
   Deploy to      : Humaix/satquest-qwen2vl-sft-v2-merged  (full model)
   Effective batch: 2 × 8 = 16


## 📂 Step 3 — Mount Drive & Get Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")

Mounted at /content/drive
✅ Drive mounted


In [ ]:
# Download HRVQA satellite images
# Images are stored as /content/HRVQA/images/{image_id}.png  e.g. 4891.png
import os
if not os.path.exists("/content/HRVQA"):
    !git clone https://huggingface.co/datasets/JNIC1/HRVQA
else:
    print("HRVQA already downloaded")

imgs = os.listdir("/content/HRVQA/images")
print(f"\n✅ HRVQA images: {len(imgs):,}")
print(f"   Sample names : {imgs[:5]}")

Cloning into 'HRVQA'...
remote: Enumerating objects: 10134, done.
remote: Total 10134 (delta 0), reused 0 (delta 0), pack-reused 10134 (from 1)
Receiving objects: 100% (10134/10134), 1.93 MiB | 17.48 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Updating files: 100% (10006/10006), done.
Filtering content: 100% (10003/10003), 15.74 GiB | 21.18 MiB/s, done.

✅ HRVQA images: 9,999
   Sample names : ['6345.png', '2253.png', '958.png', '2695.png', '8086.png']


In [ ]:
# Load SFT dataset from Google Drive
import os, shutil, zipfile

# Try zip first, then individual file, then manual upload
DRIVE_ZIP  = "your path/satquest_sft_output.zip"
DRIVE_FULL = "your path/satquest_sft_output/satquest_sft_full.json"

if os.path.exists(DRIVE_ZIP):
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall("/content")
    FULL_JSON = "/content/satquest_sft_output/satquest_sft_full.json"
    print("✅ Extracted from zip")
elif os.path.exists(DRIVE_FULL):
    os.makedirs("/content/satquest_sft_output", exist_ok=True)
    shutil.copy(DRIVE_FULL, "/content/satquest_sft_output/satquest_sft_full.json")
    FULL_JSON = "/content/satquest_sft_output/satquest_sft_full.json"
    print("✅ Copied from Drive")
else:
    from google.colab import files
    print("Uploading satquest_sft_full.json manually...")
    uploaded = files.upload()
    FULL_JSON = list(uploaded.keys())[0]

print(f"\nSFT file: {FULL_JSON}")
print(f"Exists  : {os.path.exists(FULL_JSON)}")

✅ Extracted from zip

SFT file: /content/satquest_sft_output/satquest_sft_full.json
Exists  : True


In [ ]:
# Quick validation of the SFT dataset
import json

with open(FULL_JSON, 'r') as f:
    full_records = json.load(f)

r0 = full_records[0]
ids = [r['image_id'] for r in full_records]
full_sents = sum(1 for r in full_records if len(r['verbalized_answer'].split()) >= 5)

print("=" * 55)
print("  SFT DATASET VALIDATION")
print("=" * 55)
print(f"  Total records       : {len(full_records):,}")
print(f"  Unique images       : {len(set(ids)):,}")
print(f"  image_id range      : {min(ids)} → {max(ids)}")
print(f"  image_id type       : {type(r0['image_id']).__name__} (int = good ✅)")
print(f"  Full sentences      : {full_sents:,} / {len(full_records):,} (100% ✅)")
print(f"  Fields present      : {list(r0.keys())}")
print("=" * 55)
print("\nSample record:")
print(f"  image_id         : {r0['image_id']}")
print(f"  question         : {r0['question']}")
print(f"  original_answer  : {r0['original_answer']}")
print(f"  verbalized_answer: {r0['verbalized_answer']}")
print(f"\n✅ SFT file is valid and ready for training!")

  SFT DATASET VALIDATION
  Total records       : 20,000
  Unique images       : 8,785
  image_id range      : 1 → 9998
  image_id type       : int (int = good ✅)
  Full sentences      : 20,000 / 20,000 (100% ✅)
  Fields present      : ['question_id', 'image_id', 'image_name', 'question_type', 'question', 'original_answer', 'verbalized_answer', 'image_path', 'verbalization_method']

Sample record:
  image_id         : 1
  question         : What kind of transportation can people use in this image?
  original_answer  : car
  verbalized_answer: The image shows car as the primary transportation feature.

✅ SFT file is valid and ready for training!


## 🤖 Step 4 — Load Model from  Account

> ⚠️ `xxxxx/qwen2vl-2b-hrvqa-lora` is **already a PEFT model** (has LoRA adapters baked in).  
> We load it directly — **no `get_peft_model()` call** needed (that would error).

In [ ]:
from unsloth import FastVisionModel
import torch

print(f"Loading: {SOURCE_MODEL}")
print("Downloading from HuggingFace...")

model, tokenizer = FastVisionModel.from_pretrained(
    SOURCE_MODEL,
    load_in_4bit               = True,
    use_gradient_checkpointing = "unsloth",
)

from peft import PeftModel
is_peft = isinstance(model, PeftModel)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print(f"\n✅ Model loaded")
print(f"   Type        : {type(model).__name__}")
print(f"   Is PEFT     : {is_peft}  ← True means LoRA already inside")
print(f"   Trainable   : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading: Abdullah-123/qwen2vl-2b-hrvqa-lora
==((====))==  Unsloth 2026.6.2: Fast Qwen2_Vl patching. Transformers: 4.57.1.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


...

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/116M [00:00<?, ?B/s]


✅ Model loaded
   Type        : PeftModelForCausalLM
   Is PEFT     : True  ← True means LoRA already inside
   Trainable   : 28,950,528 / 1,268,265,472 (2.28%)


## 📊 Step 5 — Build Training Dataset

In [ ]:
# Resolve image paths: /content/HRVQA/images/{image_id}.png
valid_records = []
missing = 0

for rec in full_records:
    img_path = os.path.join(IMAGE_DIR, f"{rec['image_id']}.png")
    if os.path.exists(img_path):
        rec['resolved_path'] = img_path
        valid_records.append(rec)
    else:
        missing += 1

print(f"✅ Valid records : {len(valid_records):,}")
print(f"⚠️  Missing image: {missing:,}")

if len(valid_records) == 0:
    sample_files = os.listdir(IMAGE_DIR)[:5]
    print(f"\n❌ No matches! Files in {IMAGE_DIR}: {sample_files}")
    print(f"   Expected: {full_records[0]['image_id']}.png")
    raise RuntimeError("Image path mismatch — check IMAGE_DIR")

print(f"   First path: {valid_records[0]['resolved_path']}")

✅ Valid records : 20,000
⚠️  Missing image: 0
   First path: /content/HRVQA/images/1.png


In [ ]:
from PIL import Image as PILImage
from datasets import Dataset
import random

SYSTEM_PROMPT = (
    "You are SatQuest, an expert satellite imagery analyst. "
    "When given a satellite image and a question, answer clearly "
    "in a complete, natural English sentence."
)

def make_conversation(rec):
    try:
        image = PILImage.open(rec['resolved_path']).convert("RGB")
    except Exception:
        return None
    return {
        "messages": [
            {"role": "system",
             "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user",
             "content": [{"type": "image", "image": image},
                         {"type": "text",  "text": rec['question']}]},
            {"role": "assistant",
             "content": [{"type": "text", "text": rec['verbalized_answer']}]},
        ]
    }

print("Building training samples (loading images)...")
all_samples, errors = [], 0
for i, rec in enumerate(valid_records):
    s = make_conversation(rec)
    if s: all_samples.append(s)
    else: errors += 1
    if (i+1) % 2000 == 0:
        print(f"  {i+1:,}/{len(valid_records):,}  errors: {errors}")

print(f"\n✅ {len(all_samples):,} samples built  (errors: {errors})")

# 95% train / 5% val split
random.seed(SEED)
random.shuffle(all_samples)
split = int(0.95 * len(all_samples))
train_data, val_data = all_samples[:split], all_samples[split:]
train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)
print(f"Train: {len(train_dataset):,}  |  Val: {len(val_dataset):,}")

Building training samples (loading images)...
  2,000/20,000  errors: 0
  4,000/20,000  errors: 0
  6,000/20,000  errors: 0
  8,000/20,000  errors: 0
  10,000/20,000  errors: 0
  12,000/20,000  errors: 0
  14,000/20,000  errors: 0
  16,000/20,000  errors: 0
  18,000/20,000  errors: 0
  20,000/20,000  errors: 0

✅ 20,000 samples built  (errors: 0)


KeyboardInterrupt: 

In [ ]:
from datasets import Dataset
import random

SYSTEM_PROMPT = (
    "You are SatQuest, an expert satellite imagery analyst. "
    "When given a satellite image and a question, answer clearly "
    "in a complete, natural English sentence."
)

def make_conversation(rec):
    return {
        "messages": [
            {"role": "system",
             "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user",
             "content": [{"type": "image", "image": rec['resolved_path']},
                         {"type": "text",  "text": rec['question']}]},
            {"role": "assistant",
             "content": [{"type": "text", "text": rec['verbalized_answer']}]},
        ]
    }

print("Building training samples (fast lazy loading)...")
all_samples, errors = [], 0
for i, rec in enumerate(valid_records):
    s = make_conversation(rec)
    if s: all_samples.append(s)
    else: errors += 1
    if (i+1) % 2000 == 0:
        print(f"  {i+1:,}/{len(valid_records):,}  errors: {errors}")

print(f"\n✅ {len(all_samples):,} samples built")

# 95% train / 5% val split
random.seed(SEED)
random.shuffle(all_samples)
split = int(0.95 * len(all_samples))
train_data, val_data = all_samples[:split], all_samples[split:]
train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)
print(f"Train: {len(train_dataset):,}  |  Val: {len(val_dataset):,}")


Building training samples (fast lazy loading)...
  2,000/20,000  errors: 0
  4,000/20,000  errors: 0
  6,000/20,000  errors: 0
  8,000/20,000  errors: 0
  10,000/20,000  errors: 0
  12,000/20,000  errors: 0
  14,000/20,000  errors: 0
  16,000/20,000  errors: 0
  18,000/20,000  errors: 0
  20,000/20,000  errors: 0

✅ 20,000 samples built
Train: 19,000  |  Val: 1,000


## 🏋️ Step 6 — Train (Continued SFT)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

FastVisionModel.for_training(model)  # switch to train mode

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    max_steps                   = MAX_STEPS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = LR_SCHEDULER,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    bf16                        = BF16,
    fp16                        = False,
    max_grad_norm               = MAX_GRAD_NORM,
    logging_steps               = LOGGING_STEPS,
    save_steps                  = SAVE_STEPS,
    eval_steps                  = EVAL_STEPS,
    eval_strategy               = "steps",        # <--- FIXED HERE
    save_strategy               = "steps",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    save_total_limit            = 2,
    seed                        = SEED,
    remove_unused_columns       = False,
    dataset_text_field          = "",
    max_seq_length              = MAX_SEQ_LEN,
    dataset_kwargs              = {"skip_prepare_dataset": True},
    report_to                   = "none",
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    args          = training_args,
)

steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f"✅ Trainer ready")
print(f"   Steps/epoch  : {steps_per_epoch:,}")
print(f"   Total steps  : {steps_per_epoch * NUM_EPOCHS:,}")


Unsloth: Model does not have a default image size - using 512
✅ Trainer ready
   Steps/epoch  : 1,187
   Total steps  : 2,374


In [ ]:
import time
print("🚀 Starting continued SFT...")
t0 = time.time()

train_result = trainer.train()

elapsed = time.time() - t0
print(f"\n✅ Training done — {elapsed/3600:.1f} hours")
print(f"   Final train loss : {train_result.training_loss:.4f}")
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

🚀 Starting continued SFT...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,000 | Num Epochs = 2 | Total steps = 2,376
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 28,950,528 of 2,237,936,128 (1.29% trained)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along wit

Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bits

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
200,0.072300,0.072688
400,0.073600,0.069665
600,0.075800,0.069006
800,0.063200,0.069007
1000,0.071300,0.068845
1200,0.066800,0.068493
1400,0.062200,0.068343
1600,0.065300,0.068105
1800,0.065200,0.067915
2000,0.068200,0.067736


KeyboardInterrupt: 

## 💾 Step 7 — Save Locally & Backup to Drive

In [ ]:
import shutil

# Save adapter locally in Colab
os.makedirs(ADAPTER_SAVE, exist_ok=True)
model.save_pretrained(ADAPTER_SAVE)
tokenizer.save_pretrained(ADAPTER_SAVE)
print(f"✅ Adapter saved  : {ADAPTER_SAVE}")
print(f"   Files          : {os.listdir(ADAPTER_SAVE)}")

# Backup to Google Drive (important — Colab resets on disconnect)
os.makedirs(DRIVE_BACKUP, exist_ok=True)
for fname in os.listdir(ADAPTER_SAVE):
    shutil.copy(os.path.join(ADAPTER_SAVE, fname),
                os.path.join(DRIVE_BACKUP, fname))
print(f"✅ Drive backup   : {DRIVE_BACKUP}")

✅ Adapter saved  : /content/satquest_adapter
   Files          : ['chat_template.jinja', 'adapter_config.json', 'vocab.json', 'tokenizer.json', 'adapter_model.safetensors', 'merges.txt', 'added_tokens.json', 'video_preprocessor_config.json', 'special_tokens_map.json', 'preprocessor_config.json', 'tokenizer_config.json', 'README.md']
✅ Drive backup   : /content/drive/MyDrive/satquest_continued_sft_adapter


## 🚀 Step 8 — Deploy to YOUR HuggingFace Account

**Before running this cell, replace the token:**
1. Go to → **https://huggingface.co/settings/tokens**
2. Click **New token** → Type: **Write** → Copy
3. Paste into `HF_TOKEN` in Step 2 (Configuration cell) above

Two repos will be created on **your** account:

| Repo | URL | Size | Use |
|------|-----|------|-----|
| Adapter | `your account/satquest-qwen2vl-sft-v2` | ~116 MB | Fine-tuning experiments |
| Merged | `your account/satquest-qwen2vl-sft-v2-merged` | ~4 GB | **FastAPI backend** |

In [ ]:
# ── Login to YOUR HuggingFace account ───────────────────────────────────────
from huggingface_hub import login, HfApi

login(token=HF_TOKEN)

api      = HfApi()
me       = api.whoami(token=HF_TOKEN)
print(f"✅ Logged in as : {me['name']}")

if me['name'].lower() != HF_USERNAME.lower():
    print(f"⚠️  WARNING: Logged in as '{me['name']}' but expected '{HF_USERNAME}'")
    print("   Update HF_TOKEN in Step 2 with YOUR token")
else:
    print(f"   Repos will be created at: https://huggingface.co/{HF_USERNAME}")

✅ Logged in as : Humaix
   Repos will be created at: https://huggingface.co/Humaix


In [ ]:
# ── Push LoRA Adapter (~116 MB) ──────────────────────────────────────────────
# Loads with: FastVisionModel.from_pretrained('Humaix/satquest-qwen2vl-sft-v2')

if UPLOAD_ADAPTER:
    ADAPTER_REPO = f"{HF_USERNAME}/{HF_REPO_NAME}"
    print(f"Uploading LoRA adapter → https://huggingface.co/{ADAPTER_REPO}")

    model.push_to_hub(
        ADAPTER_REPO,
        token   = HF_TOKEN,
        private = REPO_PRIVATE,
    )
    tokenizer.push_to_hub(
        ADAPTER_REPO,
        token   = HF_TOKEN,
        private = REPO_PRIVATE,
    )
    print(f"✅ Adapter live at : https://huggingface.co/{ADAPTER_REPO}")
else:
    print("Adapter upload skipped (UPLOAD_ADAPTER=False)")

Uploading LoRA adapter → https://huggingface.co/Humaix/satquest-qwen2vl-sft-v2


README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  558kB /  116MB            

Saved model to https://huggingface.co/Humaix/satquest-qwen2vl-sft-v2


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp8s3ldg7i/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

✅ Adapter live at : https://huggingface.co/Humaix/satquest-qwen2vl-sft-v2


In [ ]:
# ── Push MERGED Full Model (~4 GB) ───────────────────────────────────────────
# Merges LoRA into base weights → standalone, no base model needed
# RECOMMENDED for your FastAPI backend
# Loads with plain transformers (no Unsloth needed on server)

if UPLOAD_MERGED:
    MERGED_REPO = f"{HF_USERNAME}/{HF_REPO_NAME}-merged"
    print("Merging LoRA adapter into base weights (~5 min)...")

    # 1. Save merged model to disk
    os.makedirs(MERGED_SAVE, exist_ok=True)
    model.save_pretrained_merged(
        MERGED_SAVE,
        tokenizer,
        save_method = "merged_16bit",
    )
    print(f"✅ Merged model saved locally: {MERGED_SAVE}")
    print(f"   Size: {sum(os.path.getsize(os.path.join(MERGED_SAVE,f)) for f in os.listdir(MERGED_SAVE))/1e9:.2f} GB")

    # 2. Backup to Drive
    print("\nBacking up to Drive...")
    if os.path.exists(DRIVE_MERGED):
        shutil.rmtree(DRIVE_MERGED)
    shutil.copytree(MERGED_SAVE, DRIVE_MERGED)
    print(f"✅ Drive backup: {DRIVE_MERGED}")

    # 3. Upload to HuggingFace (~10-20 min)
    print(f"\nUploading to Hub (uploads ~4 GB — please wait 10-20 min)...")
    model.push_to_hub_merged(
        MERGED_REPO,
        tokenizer,
        save_method = "merged_16bit",
        token       = HF_TOKEN,
        private     = REPO_PRIVATE,
    )
    print(f"✅ Merged model live at: https://huggingface.co/{MERGED_REPO}")
else:
    print("Merged upload skipped (UPLOAD_MERGED=False)")

Merging LoRA adapter into base weights (~5 min)...


config.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-merged/model.safetensors:   0%|          | 15.9MB / 4.42GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:13<00:00, 73.97s/it]


Unsloth: Merge process complete. Saved to `/content/Humaix/satquest-qwen2vl-sft-v2-merged`
✅ Merged model live at: https://huggingface.co/Humaix/satquest-qwen2vl-sft-v2-merged


In [ ]:
# ── Upload Model Card (README) to your repos ────────────────────────────────

MODEL_CARD = """---
language:
- en
license: apache-2.0
base_model: Abdullah-123/qwen2vl-2b-hrvqa-lora
tags:
- satellite-imagery
- visual-question-answering
- qwen2-vl
- fine-tuned
- unsloth
- lora
datasets:
- JNIC1/HRVQA
---

# 🛰️ SatQuest — Satellite VQA (Continued SFT v2)

**Qwen2-VL-2B fine-tuned for satellite imagery question answering — produces complete natural-language sentences.**

Built on top of [Abdullah-123/qwen2vl-2b-hrvqa-lora](https://huggingface.co/Abdullah-123/qwen2vl-2b-hrvqa-lora) with continued SFT to teach sentence-style responses.

## Before vs After
```
Q: What is the main transportation?  →  Before: "car"  |  After: "The main mode of transportation visible in the image is car."
Q: Is there any water section?       →  Before: "yes"  |  After: "Yes, there is a water section present in the image."
Q: How many small vehicles?          →  Before: "2"    |  After: "There are two small vehicles visible in the image."
```

## Model Details
| Property | Value |
|----------|-------|
| Base | Qwen2-VL-2B-Instruct |
| First fine-tune | HRVQA dataset — 159,980 QA pairs |
| This fine-tune | 20,000 verbalized sentence-style QA pairs |
| Method | LoRA (rank=16) via Unsloth + TRL SFTTrainer |
| GPU | NVIDIA A100 80GB |
| Epochs | 2 (LR=1e-4, cosine) |

## Usage (with Unsloth)
```python
from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    'Humaix/satquest-qwen2vl-sft-v2',
    load_in_4bit=True,
)
FastVisionModel.for_inference(model)
```

## Usage — Merged Model (plain transformers, for production backend)
```python
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch

model = Qwen2VLForConditionalGeneration.from_pretrained(
    'Humaix/satquest-qwen2vl-sft-v2-merged',
    torch_dtype=torch.float16,
    device_map='auto',
)
processor = AutoProcessor.from_pretrained('Humaix/satquest-qwen2vl-sft-v2-merged')
```

## Project
**NED University of Engineering & Technology**
Final Year Design Project (CS-22030) — SatQuest: AI-powered Satellite Imagery Question Answering System
""".strip()

with open("/content/README.md", "w") as f:
    f.write(MODEL_CARD)

if UPLOAD_ADAPTER:
    api.upload_file(
        path_or_fileobj = "/content/README.md",
        path_in_repo    = "README.md",
        repo_id         = f"{HF_USERNAME}/{HF_REPO_NAME}",
        repo_type       = "model",
        token           = HF_TOKEN,
    )
    print(f"✅ Model card → https://huggingface.co/{HF_USERNAME}/{HF_REPO_NAME}")

if UPLOAD_MERGED:
    api.upload_file(
        path_or_fileobj = "/content/README.md",
        path_in_repo    = "README.md",
        repo_id         = f"{HF_USERNAME}/{HF_REPO_NAME}-merged",
        repo_type       = "model",
        token           = HF_TOKEN,
    )
    print(f"✅ Model card → https://huggingface.co/{HF_USERNAME}/{HF_REPO_NAME}-merged")

✅ Model card → https://huggingface.co/Humaix/satquest-qwen2vl-sft-v2
✅ Model card → https://huggingface.co/Humaix/satquest-qwen2vl-sft-v2-merged


In [ ]:
!pip install qwen-vl-utils


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 70.8 MB/s eta 0:00:00


## 🧪 Step 9 — Quick Inference Test

In [ ]:
from qwen_vl_utils import process_vision_info
from PIL import Image as PILImage


FastVisionModel.for_inference(model)

def answer(image_id: int, question: str) -> str:
    image = PILImage.open(os.path.join(IMAGE_DIR, f"{image_id}.png")).convert("RGB")
    messages = [
        {"role": "system",    "content": [{"type": "text",  "text": SYSTEM_PROMPT}]},
        {"role": "user",      "content": [{"type": "image", "image": image},
                                          {"type": "text",  "text": question}]},
    ]
    prompt       = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs       = tokenizer(text=[prompt], images=image_inputs, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False, repetition_penalty=1.1)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


print("=" * 65)
print("  INFERENCE TEST — SatQuest v2 (sentence-style answers)")
print("=" * 65)
for rec in valid_records[:5]:
    try:
        pred = answer(rec['image_id'], rec['question'])
    except Exception as e:
        pred = f"ERROR: {e}"
    print(f"\nQ        : {rec['question']}")
    print(f"Original : {rec['original_answer']}")
    print(f"Target   : {rec['verbalized_answer']}")
    print(f"Predicted: {pred}")
    print("-" * 60)


  INFERENCE TEST — SatQuest v2 (sentence-style answers)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bits


Q        : What kind of transportation can people use in this image?
Original : car
Target   : The image shows car as the primary transportation feature.
Predicted: The image shows car as the primary transportation feature.
------------------------------------------------------------

Q        : What is the main scene of this image?
Original : rural
Target   : The main scene of this image in the image is rural.
Predicted: The main scene of this image in the image is lowdensity.
------------------------------------------------------------

Q        : Is there any water section in this image?
Original : no
Target   : No, there is no any in the image.
Predicted: No, there is no any in the image.
------------------------------------------------------------

Q        : What is the main scene of this image?
Original : highdensity
Target   : The main scene of this image in the image is highdensity.
Predicted: The main scene of this image in the image is lowdensity.
--------------------------

## 🗜️ Step 10 — Download Adapter to PC (optional)

In [ ]:
# Optional: download the adapter zip to your local PC
# Skip if you already pushed to HuggingFace
import zipfile
from google.colab import files

ZIP_PATH = "/content/satquest_adapter.zip"
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for fname in os.listdir(ADAPTER_SAVE):
        z.write(os.path.join(ADAPTER_SAVE, fname), fname)

print(f"✅ Zipped: {os.path.getsize(ZIP_PATH)/1e6:.1f} MB")
files.download(ZIP_PATH)
print("📥 Download started")

---
## ✅ Final Checklist

After all cells complete, verify:

- [ ] Visit **https://huggingface.co/** — two new repos appear
- [ ] `xxx/satquest-qwen2vl-sft-v2` → **Files** tab has `adapter_model.safetensors`
- [ ] `xxxx/satquest-qwen2vl-sft-v2-merged` → **Files** tab has `model.safetensors`
- [ ] Both repos show the **Model Card** (README) with project info
- [ ] Inference test shows complete sentences (not single words)

---

## 📖 Load in Your FastAPI Backend

```python
# Use merged model — no Unsloth needed on the server
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch

MODEL_ID = "xxxxx/satquest-qwen2vl-sft-v2-merged"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
```